# BackPaS: entrenamiento paralelo por trabajo

Ejecuta una fila distinta de `configs/cloud_jobs.json` por sesión de Colab. No reutilices el mismo trabajo en dos sesiones.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/backbone-pbo'
DRIVE_ROOT = '/content/drive/MyDrive/PBO-Tesis'
JOB = 'MVC/baseline/seed-0'  # Cambiar en cada sesión


In [ ]:
# Clonar o actualizar el repositorio
import os, subprocess, shutil, tarfile, glob
if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/jgomezaTI/PBO-DA.git', REPO], check=True)
else:
    subprocess.run(['git','-C', REPO, 'pull', '--ff-only'], check=True)

# Restaurar datasets y resultados desde Drive.
DATA_ROOT = f'{DRIVE_ROOT}/datasets'
for name in ['MVC', 'CA']:
    archive = f'{DATA_ROOT}/{name}/{name}.tar.gz'
    target = f'{REPO}/data/backpas/{name}'
    if os.path.exists(archive) and not os.path.exists(target):
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(f'{REPO}/data/backpas')

# MIS se subió en partes por el límite de tamaño de Drive.
mis_dir = f'{DATA_ROOT}/MIS'
mis_archive = f'{mis_dir}/MIS.tar.gz'
parts = sorted(glob.glob(f'{mis_dir}/MIS.tar.gz.part-*'))
if not os.path.exists(mis_archive) and parts:
    with open(mis_archive, 'wb') as output:
        for part in parts:
            with open(part, 'rb') as source:
                shutil.copyfileobj(source, output)
if os.path.exists(mis_archive) and not os.path.exists(f'{REPO}/data/backpas/MIS'):
    with tarfile.open(mis_archive, 'r:gz') as tar:
        tar.extractall(f'{REPO}/data/backpas')

# Restaurar los resultados previamente entrenados.
results_archive = f'{DRIVE_ROOT}/runs/training-results.tar.gz'
if os.path.exists(results_archive):
    with tarfile.open(results_archive, 'r:gz') as tar:
        tar.extractall(REPO)

subprocess.run(['python','-m','pip','install','-e','.[ml]'], cwd=REPO, check=True)


In [ ]:
# Ejecución reanudable; cambia JOB antes de arrancar otra sesión
subprocess.run(['python','scripts/cloud_run.py','--repo',REPO,'--job',JOB,'--drive-root',DRIVE_ROOT,'--resume'], cwd=REPO, check=True)
